In [9]:
# %% [markdown]
# # Typhoon Precipitation Accumulation by MJO Phase Grouping
# 
 
 
import pandas as pd
import numpy as np
import xarray as xr
import os
from datetime import datetime

# Set paths (adjust according to actual environment)
base_path = './'
typhoon_tracks_path = os.path.join(base_path, 'typhoon_output/landfall_typhoons_tracks.csv')
mjo_path = os.path.join(base_path, 'ERA5 MJO (1950-2024).csv')
pre_base_path = './pre/'

 
# ## 1. Read MJO Data

 
print("="*60)
print("Step 1: Reading MJO data")
mjo_df = pd.read_csv(mjo_path)
mjo_df['date'] = pd.to_datetime(mjo_df['date'])
mjo_df.set_index('date', inplace=True)

# Filter for 1960-2024
mjo_df = mjo_df.loc['1960':'2024']
print(f"MJO data time range: {mjo_df.index.min()} to {mjo_df.index.max()}")
print(f"Total records: {len(mjo_df)}")

# To speed up lookup, create a set of dates (date part only)
mjo_dates_set = set(mjo_df.index.date)

 
# ## 2. Filter Typhoons Active from June to October from Track Data

# %%
print("\nStep 2: Reading and filtering typhoon track data")
tracks_df = pd.read_csv(typhoon_tracks_path)
tracks_df['TIME'] = pd.to_datetime(tracks_df['TIME'])

# Time range 1960-2024, months June-October
mask = (tracks_df['TIME'].dt.year >= 1960) & (tracks_df['TIME'].dt.year <= 2024) & \
    (tracks_df['TIME'].dt.month.between(6, 10))
filtered_tracks = tracks_df.loc[mask]

# Get unique typhoon IDs
typhoon_codes = filtered_tracks['chinese_code'].unique()
print(f"Filtered {len(typhoon_codes)} typhoons (with track points from June to October)")
print("First 10 typhoon IDs:", typhoon_codes[:10])

 
# ## 3. Get Template Latitude and Longitude (from the first existing precipitation file)

# %%
print("\nStep 3: Getting template latitude and longitude")
template_ds = None
for code in typhoon_codes:
    code_str = f"{code:04d}"
    nc_file = os.path.join(pre_base_path, f"pre_{code_str}.nc")
    if os.path.exists(nc_file):
        template_ds = xr.open_dataset(nc_file)
        lat = template_ds.lat.values
        lon = template_ds.lon.values
        print(f"Using typhoon {code_str}'s latitude and longitude as template")
        print(f"Latitude range: {lat[0]:.2f} ~ {lat[-1]:.2f}, points {len(lat)}")
        print(f"Longitude range: {lon[0]:.2f} ~ {lon[-1]:.2f}, points {len(lon)}")
        break
if template_ds is None:
    raise FileNotFoundError("Error: No precipitation files found, cannot proceed with accumulation.")

# 4. Initialize Accumulation Arrays and Counters

 
shape = (len(lat), len(lon))
acc_phase12 = np.zeros(shape, dtype=np.float64)
acc_phase34 = np.zeros(shape, dtype=np.float64)
acc_phase56 = np.zeros(shape, dtype=np.float64)
acc_phase78 = np.zeros(shape, dtype=np.float64)

cnt_phase12 = 0
cnt_phase34 = 0
cnt_phase56 = 0
cnt_phase78 = 0
total_days = 0

missing_files = []          # Record missing precipitation files
missing_mjo_dates = []       # Record dates missing in MJO data

 
# ## 5. Process Each Typhoon and Accumulate Precipitation

# %%
print("\nStep 4: Processing typhoons and accumulating precipitation")
for i, code in enumerate(typhoon_codes):
    code_str = f"{code:04d}"
    nc_file = os.path.join(pre_base_path, f"pre_{code_str}.nc")
    
    if not os.path.exists(nc_file):
        print(f"  Precipitation file not found: {nc_file}")
        missing_files.append(code_str)
        continue
    
    print(f"\nProcessing typhoon {code_str} ({i+1}/{len(typhoon_codes)})")
    ds = xr.open_dataset(nc_file)
    times = ds.time.values
    print(f"  Precipitation file contains {len(times)} time steps")
    
    for t in times:
        t_date = pd.Timestamp(t).date()   # Convert to date object
        
        # Check if date exists in MJO data
        if t_date not in mjo_dates_set:
            missing_mjo_dates.append(t_date)
            continue
        
        mjo_row = mjo_df.loc[pd.Timestamp(t_date)]   # May return Series or DataFrame
        if isinstance(mjo_row, pd.DataFrame):
            mjo_row = mjo_row.iloc[0]   # Take the first row (theoretically only one)
        
        amplitude = mjo_row['amplitude']
        if amplitude < 1.0:
            continue    # Skip dates with amplitude < 1
        
        phase = int(mjo_row['phase'])   # Ensure integer
        # Grouping
        if phase in (1, 2):
            group = '12'
        elif phase in (3, 4):
            group = '34'
        elif phase in (5, 6):
            group = '56'
        elif phase in (7, 8):
            group = '78'
        else:
            print(f"    Warning: phase={phase} out of 1-8, skipping")
            continue
        
        # Read precipitation data for this time step (2D)
        prec_data = ds.prec.sel(time=t).values   # Equivalent to isel(time=idx)
        # Convert nan to 0 before accumulation to avoid contaminating the array
        prec_clean = np.nan_to_num(prec_data, nan=0.0)
        
        # Accumulate to the corresponding group
        if group == '12':
            acc_phase12 += prec_clean
            cnt_phase12 += 1
        elif group == '34':
            acc_phase34 += prec_clean
            cnt_phase34 += 1
        elif group == '56':
            acc_phase56 += prec_clean
            cnt_phase56 += 1
        elif group == '78':
            acc_phase78 += prec_clean
            cnt_phase78 += 1
        
        total_days += 1
    
    ds.close()
    
    # Print current accumulated days after each typhoon
    print(f"  Current accumulated days: group12={cnt_phase12}, group34={cnt_phase34}, group56={cnt_phase56}, group78={cnt_phase78}")

# 6. Output Statistics

# %%
print("\n" + "="*60)
print("Processing complete!")
print(f"Total valid days (amplitude≥1): {total_days}")
print(f"Group12 (phase 1-2): {cnt_phase12} days")
print(f"Group34 (phase 3-4): {cnt_phase34} days")
print(f"Group56 (phase 5-6): {cnt_phase56} days")
print(f"Group78 (phase 7-8): {cnt_phase78} days")

if missing_files:
    print(f"\nMissing precipitation files for typhoon IDs (total {len(missing_files)}): {missing_files}")
if missing_mjo_dates:
    # Remove duplicates and print first 10
    unique_missing = set(missing_mjo_dates)
    print(f"\nDates missing in MJO data (total {len(unique_missing)}), e.g.: {sorted(unique_missing)[:10]}")

# %% [markdown]
# ## 7. Save Accumulated Results as NC Files

# %%
def save_accumulated_nc(data, group_name, lat, lon, ref_time):
    """
    Save accumulated array as NetCDF file, with time dimension fixed to ref_time
    """
    ds_out = xr.Dataset(
        {
            'prec': (('time', 'lat', 'lon'), data[np.newaxis, :, :])
        },
        coords={
            'time': ref_time,
            'lat': lat,
            'lon': lon
        }
    )
    ds_out.prec.attrs['units'] = 'mm'   # Assuming precipitation unit is mm
    ds_out.prec.attrs['long_name'] = 'Accumulated precipitation by MJO phase group'
    out_filename = f"01MJOpre_phase{group_name}.nc"
    ds_out.to_netcdf(out_filename)
    print(f"Saved: {out_filename}")

# Set a fixed reference time
ref_time = np.array(['2000-01-01'], dtype='datetime64[ns]')

print("\nSaving result files:")
save_accumulated_nc(acc_phase12, '12', lat, lon, ref_time)
save_accumulated_nc(acc_phase34, '34', lat, lon, ref_time)
save_accumulated_nc(acc_phase56, '56', lat, lon, ref_time)
save_accumulated_nc(acc_phase78, '78', lat, lon, ref_time)

print("\nAll processing complete!")

Step 1: Reading MJO data
MJO data time range: 1960-01-01 00:00:00 to 2024-12-31 00:00:00
Total records: 23742

Step 2: Reading and filtering typhoon track data
Filtered 471 typhoons (with track points from June to October)
First 10 typhoon IDs: [6001 6005 6007 6008 6012 6014 6016 6024 6104 6109]

Step 3: Getting template latitude and longitude
Using typhoon 6001's latitude and longitude as template
Latitude range: 18.05 ~ 53.95, points 360
Longitude range: 72.05 ~ 135.95, points 640

Step 4: Processing typhoons and accumulating precipitation

Processing typhoon 6001 (1/471)
  Precipitation file contains 16 time steps
  Current accumulated days: group12=1, group34=0, group56=0, group78=4

Processing typhoon 6005 (2/471)
  Precipitation file contains 16 time steps
  Current accumulated days: group12=5, group34=1, group56=1, group78=4

Processing typhoon 6007 (3/471)
  Precipitation file contains 11 time steps
  Current accumulated days: group12=5, group34=2, group56=7, group78=4

Process

In [8]:
#!/usr/bin/env python
# coding: utf-8

 
# Typhoon Precipitation Accumulation by MJO Phase and Typhoon Category
 

import pandas as pd
import numpy as np
import xarray as xr
import os
from tqdm import tqdm

# ==================== 1. Load Data ====================
print("=" * 60)
print("Step 1: Load typhoon landfall information and MJO data")
print("=" * 60)

# Typhoon landfall information
landfall_info = pd.read_csv('./typhoon_output/landfall_typhoons_info.csv')
print(f"Total typhoon landfall records: {len(landfall_info)}")
print(f"Typhoon ID range: {landfall_info['chinese_code'].min()} ~ {landfall_info['chinese_code'].max()}")

# MJO data
mjo_df = pd.read_csv('./ERA5 MJO (1950-2024).csv')
mjo_df['date'] = pd.to_datetime(mjo_df['date'])
mjo_df.set_index('date', inplace=True)
print(f"MJO data time range: {mjo_df.index.min()} to {mjo_df.index.max()}")

# ==================== 2. Build MJO Dictionary (amplitude ≥ 1 only) ====================
print("\n" + "=" * 60)
print("Step 2: Build MJO dictionary (amplitude ≥ 1)")
print("=" * 60)

mjo_dict = {}
for date, row in mjo_df.iterrows():
    if row['amplitude'] >= 1.0:
        # phase might be float, round to integer
        phase = int(round(row['phase']))
        if 1 <= phase <= 8:
            mjo_dict[date.strftime('%Y-%m-%d')] = phase

print(f"Qualified dates count: {len(mjo_dict)}")
print("First 5 examples:", list(mjo_dict.items())[:5])

# ==================== 3. Extract Maximum Typhoon Category ====================
print("\n" + "=" * 60)
print("Step 3: Extract maximum typhoon category (max_wind_category)")
print("=" * 60)

# For typhoons with multiple landfall records, take the maximum to ensure consistency
typhoon_category = landfall_info.groupby('chinese_code')['max_wind_category'].max().to_dict()
print(f"Unique typhoon count (deduplicated by ID): {len(typhoon_category)}")

# Count typhoons by category
cat_counts = pd.Series(typhoon_category.values()).value_counts().sort_index()
for cat in [1, 2, 3]:
    name = ['Storms', 'TCs', 'Super TCs'][cat-1]
    print(f"Category {cat} ({name}): {cat_counts.get(cat, 0)} typhoons")
other_cats = [c for c in cat_counts.index if c not in [1,2,3]]
if other_cats:
    print(f"Other categories {other_cats} will be skipped")

# ==================== 4. Get Precipitation Grid Information ====================
print("\n" + "=" * 60)
print("Step 4: Get precipitation grid coordinates (from first available precipitation file)")
print("=" * 60)

pre_base_dir = './pre/'
first_grid = None
for code in typhoon_category.keys():
    file_path = os.path.join(pre_base_dir, f"pre_{code:04d}.nc")
    if os.path.exists(file_path):
        try:
            with xr.open_dataset(file_path) as ds:
                lat = ds['lat'].values
                lon = ds['lon'].values
                print(f"Successfully obtained grid from typhoon {code}:")
                print(f"  lat: {len(lat)} points, range {lat[0]:.2f} ~ {lat[-1]:.2f}")
                print(f"  lon: {len(lon)} points, range {lon[0]:.2f} ~ {lon[-1]:.2f}")
                first_grid = (lat, lon)
                break
        except Exception as e:
            print(f"Error reading {file_path}: {e}, trying next")
            continue
if first_grid is None:
    raise FileNotFoundError("No available precipitation files found, cannot determine grid coordinates. Program exiting.")
lat, lon = first_grid

# ==================== 5. Initialize Accumulation Arrays and Counters ====================
print("\n" + "=" * 60)
print("Step 5: Initialize accumulation arrays (phase groups × category groups × lat × lon)")
print("=" * 60)

# Grouping notes:
# Phase group index 0 → (1,2), 1 → (3,4), 2 → (5,6), 3 → (7,8)
# Category group index 0 → 1 (Storms), 1 → 2 (TCs), 2 → 3 (Super TCs)

accum_precip = np.zeros((4, 3, len(lat), len(lon)), dtype=np.float64)
count_precip = np.zeros((4, 3), dtype=int)

def get_phase_group(phase):
    """Return group index based on phase"""
    if phase in [1, 2]:
        return 0
    elif phase in [3, 4]:
        return 1
    elif phase in [5, 6]:
        return 2
    elif phase in [7, 8]:
        return 3
    else:
        return None

# ==================== 6. Process Each Typhoon and Accumulate Precipitation ====================
print("\n" + "=" * 60)
print("Step 6: Process each typhoon precipitation file (replace NaN with 0)")
print("=" * 60)

missing_files = []          # Missing precipitation files
total_processed_days = 0    # Total processed precipitation days

# Use tqdm for progress bar
for code, cat in tqdm(typhoon_category.items(), desc="Processing typhoons"):
    # Only process categories 1-3
    if cat not in [1, 2, 3]:
        continue

    file_path = os.path.join(pre_base_dir, f"pre_{code:04d}.nc")
    if not os.path.exists(file_path):
        missing_files.append(code)
        continue

    try:
        ds = xr.open_dataset(file_path)
    except Exception as e:
        print(f"Failed to open {file_path}: {e}")
        missing_files.append(code)
        continue

    # Get time coordinates
    times = ds['time'].values
    dates = pd.DatetimeIndex(times)

    # Filter indices for months 6-10
    month_mask = (dates.month >= 6) & (dates.month <= 10)
    valid_indices = np.where(month_mask)[0]

    if len(valid_indices) == 0:
        ds.close()
        continue

    cat_group = cat - 1   # Category group index

    # Process each valid date
    for idx in valid_indices:
        dt = dates[idx]
        date_str = dt.strftime('%Y-%m-%d')

        # Check MJO condition
        if date_str not in mjo_dict:
            continue
        phase = mjo_dict[date_str]
        phase_group = get_phase_group(phase)
        if phase_group is None:
            continue   # Abnormal phase (theoretically shouldn't happen)

        # Extract precipitation field and replace NaN with 0 (avoid contamination)
        precip_2d = ds['prec'].isel(time=idx).values
        precip_2d = np.nan_to_num(precip_2d, nan=0.0)

        # Accumulate
        accum_precip[phase_group, cat_group, :, :] += precip_2d
        count_precip[phase_group, cat_group] += 1
        total_processed_days += 1

    ds.close()

print(f"\nProcessing complete! Total processed precipitation days: {total_processed_days}")

# ==================== 7. Print Missing Files ====================
print("\n" + "=" * 60)
print("Step 7: Missing precipitation files list")
print("=" * 60)

if missing_files:
    print(f"Total missing precipitation files: {len(missing_files)}")
    for code in missing_files:
        print(f"  pre_{code:04d}.nc")
else:
    print("All typhoon precipitation files exist.")

# ==================== 8. Output Accumulation Statistics ====================
print("\n" + "=" * 60)
print("Step 8: Accumulation statistics by group (precipitation days)")
print("=" * 60)

phase_names = ['Phases 1-2', 'Phases 3-4', 'Phases 5-6', 'Phases 7-8']
cat_names = ['Storms', 'TCs', 'Super TCs']

for p in range(4):
    for c in range(3):
        num = count_precip[p, c]
        if num > 0:
            print(f"{phase_names[p]} - {cat_names[c]}: {num} precipitation days")

# Summarize by phase group
for p in range(4):
    total = count_precip[p, :].sum()
    print(f"{phase_names[p]} total: {total} precipitation days")

# ==================== 9. Save Results as NetCDF Files ====================
print("\n" + "=" * 60)
print("Step 9: Save accumulation results as NetCDF files (filename prefix 02)")
print("=" * 60)

# Reference time (arbitrary, only for coordinates)
ref_time = np.datetime64('2000-01-01')

for p in range(4):
    for c in range(3):
        data = accum_precip[p, c, :, :]
        # Save even if no data (all-zero array) for processing consistency
        # Uncomment next line to skip all-zero groups
        # if np.all(data == 0): continue

        # Create Dataset
        ds_out = xr.Dataset(
            {
                'prec': (('time', 'lat', 'lon'), data[np.newaxis, :, :])
            },
            coords={
                'time': [ref_time],
                'lat': lat,
                'lon': lon
            },
            attrs={
                'description': f'Accumulated precipitation for {phase_names[p]} and {cat_names[c]}',
                'phase_group': p+1,
                'category_group': c+1,
                'num_days': int(count_precip[p, c]),
                'units': 'mm (accumulated)',
                'creation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
            }
        )

        # Save filename
        filename = f"02_phase{p+1}_cat{c+1}.nc"
        ds_out.to_netcdf(filename)
        print(f"Saved {filename} (accumulation days: {count_precip[p, c]})")

print("\nAll processing complete!")

Step 1: Load typhoon landfall information and MJO data
Total typhoon landfall records: 490
Typhoon ID range: 4 ~ 9914
MJO data time range: 1950-01-01 00:00:00 to 2024-12-31 00:00:00

Step 2: Build MJO dictionary (amplitude ≥ 1)
Qualified dates count: 16646
First 5 examples: [('1950-01-05', 1), ('1950-01-06', 1), ('1950-01-14', 4), ('1950-01-15', 4), ('1950-01-16', 4)]

Step 3: Extract maximum typhoon category (max_wind_category)
Unique typhoon count (deduplicated by ID): 490
Category 1 (Storms): 171 typhoons
Category 2 (TCs): 205 typhoons
Category 3 (Super TCs): 114 typhoons

Step 4: Get precipitation grid coordinates (from first available precipitation file)
Successfully obtained grid from typhoon 4:
  lat: 360 points, range 18.05 ~ 53.95
  lon: 640 points, range 72.05 ~ 135.95

Step 5: Initialize accumulation arrays (phase groups × category groups × lat × lon)

Step 6: Process each typhoon precipitation file (replace NaN with 0)


Processing typhoons: 100%|█████████████████████████████████████████████████████████████████| 490/490 [00:14<00:00, 33.26it/s]



Processing complete! Total processed precipitation days: 2301

Step 7: Missing precipitation files list
All typhoon precipitation files exist.

Step 8: Accumulation statistics by group (precipitation days)
Phases 1-2 - Storms: 119 precipitation days
Phases 1-2 - TCs: 179 precipitation days
Phases 1-2 - Super TCs: 83 precipitation days
Phases 3-4 - Storms: 98 precipitation days
Phases 3-4 - TCs: 134 precipitation days
Phases 3-4 - Super TCs: 102 precipitation days
Phases 5-6 - Storms: 278 precipitation days
Phases 5-6 - TCs: 475 precipitation days
Phases 5-6 - Super TCs: 314 precipitation days
Phases 7-8 - Storms: 136 precipitation days
Phases 7-8 - TCs: 236 precipitation days
Phases 7-8 - Super TCs: 147 precipitation days
Phases 1-2 total: 381 precipitation days
Phases 3-4 total: 334 precipitation days
Phases 5-6 total: 1067 precipitation days
Phases 7-8 total: 519 precipitation days

Step 9: Save accumulation results as NetCDF files (filename prefix 02)
Saved 02_phase1_cat1.nc (accum

In [11]:
# -*- coding: utf-8 -*-
"""
Accumulate typhoon precipitation by category (considering MJO≥1 condition)
   - 03_storms.nc      (Category 1 accumulation)
   - 03_tcs.nc         (Category 2 accumulation)
   - 03_super_tcs.nc   (Category 3 accumulation)
"""

import pandas as pd
import numpy as np
import xarray as xr
import os
from datetime import datetime

# ==================== 1. Read data ====================
print("="*60)
print("1. Read raw data")
print("="*60)

# Typhoon landfall information
df_landfall = pd.read_csv('./typhoon_output/landfall_typhoons_info.csv')
print(f"Original number of landfall records: {len(df_landfall)}")

# Typhoon track information
df_tracks = pd.read_csv('./typhoon_output/landfall_typhoons_tracks.csv')
print(f"Original number of track records: {len(df_tracks)}")

# MJO data
df_mjo = pd.read_csv('./ERA5 MJO (1950-2024).csv')
print(f"MJO data records: {len(df_mjo)}")

# ==================== 2. Data preprocessing ====================
print("\n" + "="*60)
print("2. Data preprocessing")
print("="*60)

# 2.1 Filter years 1960-2024 from landfall info
df_landfall = df_landfall[(df_landfall['year'] >= 1960) & (df_landfall['year'] <= 2024)]
print(f"Landfall records kept for 1960-2024: {len(df_landfall)}")

# 2.2 Extract typhoons active in June-October from track data
df_tracks['TIME'] = pd.to_datetime(df_tracks['TIME'])
df_tracks['month'] = df_tracks['TIME'].dt.month
active_mask = (df_tracks['month'] >= 6) & (df_tracks['month'] <= 10)
active_typhoons = set(df_tracks.loc[active_mask, 'chinese_code'].unique())
print(f"Number of typhoons active in June-October in track data: {len(active_typhoons)}")

# 2.3 Filter landfall info with the active typhoon set
df_landfall = df_landfall[df_landfall['chinese_code'].isin(active_typhoons)]
print(f"Landfall records after matching active typhoons: {len(df_landfall)}")

# 2.4 Get unique typhoons and their categories (max_wind_category)
df_unique = df_landfall[['chinese_code', 'max_wind_category']].drop_duplicates(subset='chinese_code')
print(f"Number of unique typhoons: {len(df_unique)}")
print("Number of typhoons per category (by max_wind_category):")
print(df_unique['max_wind_category'].value_counts().sort_index())

# 2.5 Prepare MJO quick lookup table (indexed by date)
df_mjo['date'] = pd.to_datetime(df_mjo['date'])
df_mjo.set_index('date', inplace=True)
mjo_amp = df_mjo['amplitude']   # Series indexed by date

# ==================== 3. Initialize accumulation arrays ====================
print("\n" + "="*60)
print("3. Initialize accumulation arrays")
print("="*60)

# Accumulation arrays for three categories, initially None, created with first valid typhoon
cat_arrays = [None, None, None]        # index 0: category 1, 1: category 2, 2: category 3
cat_counts = [0, 0, 0]                 # typhoon counts per category
cat_days = [0, 0, 0]                   # total accumulated days per category
ref_lat = None                         # reference latitude
ref_lon = None                         # reference longitude
ref_time = pd.Timestamp('2000-01-01')  # placeholder time in output files

# Directory for precipitation files
pre_dir = './pre/'

# Record typhoon codes with missing files
missing_files = []

# ==================== 4. Process typhoon by typhoon ====================
print("\n" + "="*60)
print("4. Start processing by typhoon")
print("="*60)

for idx, row in df_unique.iterrows():
    code = row['chinese_code']
    cat = row['max_wind_category']
    
    # Category validity check (must be 1, 2, or 3)
    if cat not in [1,2,3]:
        print(f"Warning: Typhoon {code} has invalid category {cat}, skipping")
        continue
    
    cat_idx = cat - 1   # convert to 0,1,2 indices
    fpath = os.path.join(pre_dir, f"pre_{code:04d}.nc")
    
    print(f"\n>>> Processing typhoon {code} (Category {cat})...")
    
    # Check if precipitation file exists
    if not os.path.exists(fpath):
        print(f"  File not found: {fpath}")
        missing_files.append(code)
        continue
    
    # Open precipitation file
    try:
        ds = xr.open_dataset(fpath)
    except Exception as e:
        print(f"  Failed to open file: {e}")
        missing_files.append(code)
        continue
    
    # Get time coordinates and convert to dates (truncate time part)
    times = pd.to_datetime(ds.time.values)
    dates = times.floor('D')   # truncate to date
    
    # Select dates where MJO amplitude >= 1
    valid_indices = []
    valid_dates = []
    for i, d in enumerate(dates):
        amp = mjo_amp.get(d, 0)   # if date not in MJO, return 0 (condition not satisfied)
        if amp >= 1:
            valid_indices.append(i)
            valid_dates.append(d)
    
    if len(valid_indices) == 0:
        print(f"  No dates with MJO≥1, skipping")
        ds.close()
        continue
    
    print(f"  Number of dates meeting MJO condition: {len(valid_indices)}")
    cat_days[cat_idx] += len(valid_indices)
    
    # Extract precipitation fields for these dates
    prec_sel = ds.prec.isel(time=valid_indices)   # shape: (n_time, lat, lon)
    
    # Compute sum over valid grid points (ignoring NaN)
    # Method: nansum, then set all-NaN grid points back to NaN via valid count
    prec_sum = np.nansum(prec_sel.values, axis=0)                 # (lat, lon)
    valid_grid = np.sum(~np.isnan(prec_sel.values), axis=0) > 0   # grid points with at least one valid value
    prec_sum[~valid_grid] = np.nan
    
    # Get lat/lon coordinates (assume all typhoons share the same grid)
    lat = ds.lat.values
    lon = ds.lon.values
    
    # If first time for this category, initialize accumulation array and save reference coordinates
    if cat_arrays[cat_idx] is None:
        cat_arrays[cat_idx] = np.full((len(lat), len(lon)), np.nan, dtype=np.float32)
        if ref_lat is None:   # save global reference from first valid typhoon
            ref_lat = lat
            ref_lon = lon
            print(f"  Reference grid initialized: lat range [{lat.min():.2f}, {lat.max():.2f}], lon range [{lon.min():.2f}, {lon.max():.2f}]")
    else:
        # Check grid consistency (simple shape and endpoint comparison; could be strengthened)
        if len(lat) != len(ref_lat) or len(lon) != len(ref_lon) or \
           not np.allclose(lat[[0,-1]], ref_lat[[0,-1]]) or \
           not np.allclose(lon[[0,-1]], ref_lon[[0,-1]]):
            print(f"  Error: Grid of typhoon {code} inconsistent with reference grid, skipping")
            ds.close()
            continue
    
    # Accumulate into the corresponding category array
    arr = cat_arrays[cat_idx]          # current category accumulation array
    mask_valid = ~np.isnan(prec_sum)    # grid points with valid precipitation this time
    mask_arr_nan = np.isnan(arr)        # grid points currently NaN in the accumulation array
    
    # Case 1: accumulation array is NaN and this time valid → assign directly
    arr[mask_valid & mask_arr_nan] = prec_sum[mask_valid & mask_arr_nan]
    # Case 2: accumulation array already has a value and this time valid → accumulate
    both_valid = mask_valid & ~mask_arr_nan
    arr[both_valid] += prec_sum[both_valid]
    
    cat_counts[cat_idx] += 1
    ds.close()
    print(f"  Accumulation done. Category {cat} cumulative typhoons: {cat_counts[cat_idx]}, cumulative days: {cat_days[cat_idx]}")

# ==================== 5. Missing file statistics ====================
print("\n" + "="*60)
print("5. Missing file statistics")
print("="*60)
if missing_files:
    print(f"Typhoon codes with missing precipitation files (total {len(missing_files)}):")
    print(missing_files)
else:
    print("All typhoons have precipitation files.")

# ==================== 6. Save accumulation results to NetCDF ====================
print("\n" + "="*60)
print("6. Save accumulation results to NetCDF")
print("="*60)

# Category name mapping
cat_names = {0: 'storms', 1: 'tcs', 2: 'super_tcs'}

for i, arr in enumerate(cat_arrays):
    if arr is None:
        print(f"Category {i+1} has no data, skipping save")
        continue
    
    # Build xarray Dataset
    ds_out = xr.Dataset(
        {
            'prec': (('lat', 'lon'), arr.astype(np.float32))
        },
        coords={
            'lat': ref_lat,
            'lon': ref_lon,
            'time': ref_time
        }
    )
    # Add time dimension for consistency with other data
    ds_out = ds_out.expand_dims('time')
    
    # Set encoding, _FillValue as NaN
    encoding = {'prec': {'dtype': 'float32', '_FillValue': np.nan}}
    
    filename = f"03_{cat_names[i]}.nc"
    ds_out.to_netcdf(filename, encoding=encoding)
    print(f"  Saved: {filename}")

# ==================== 7. Final statistics ====================
print("\n" + "="*60)
print("7. Final statistics")
print("="*60)
print(f"Category 1 (Storms)   : typhoons {cat_counts[0]}, total days {cat_days[0]}")
print(f"Category 2 (TCs)       : typhoons {cat_counts[1]}, total days {cat_days[1]}")
print(f"Category 3 (Super TCs) : typhoons {cat_counts[2]}, total days {cat_days[2]}")
print("\nProcessing complete!")

1. Read raw data
Original number of landfall records: 490
Original number of track records: 16526
MJO data records: 27394

2. Data preprocessing
Landfall records kept for 1960-2024: 490
Number of typhoons active in June-October in track data: 471
Landfall records after matching active typhoons: 471
Number of unique typhoons: 471
Number of typhoons per category (by max_wind_category):
1    167
2    192
3    112
Name: max_wind_category, dtype: int64

3. Initialize accumulation arrays

4. Start processing by typhoon

>>> Processing typhoon 6001 (Category 2)...
  Number of dates meeting MJO condition: 5
  Reference grid initialized: lat range [18.05, 53.95], lon range [72.05, 135.95]
  Accumulation done. Category 2 cumulative typhoons: 1, cumulative days: 5

>>> Processing typhoon 6005 (Category 3)...
  Number of dates meeting MJO condition: 6
  Accumulation done. Category 3 cumulative typhoons: 1, cumulative days: 6

>>> Processing typhoon 6007 (Category 3)...
  Number of dates meeting MJ